# 04 · Geração fundamentada: JSON válido, citação da fonte e recusa

O RAG simples (config B) recupera os 3 trechos mais próximos e pede ao LLM que responda **somente** com eles, em JSON:

```json
{"resposta": "...", "trecho_ids": ["T11"], "recusou": false}
```
O Ollama força o formato com *structured outputs* (JSON Schema). Mesmo assim **validamos** o JSON e as citações.

In [1]:
import sys, json
from pathlib import Path
RAIZ = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(RAIZ))
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
pd.set_option("display.max_colwidth", 140); pd.set_option("display.width", 200)
from rotalivre_ai import config, kb, metrics, montar, pipelines
from rotalivre_ai.runner import checar_ollama, rodar_todas
print(checar_ollama())

Ollama OK em http://localhost:11434 | LLM=qwen2.5:7b | embeddings=bge-m3


In [2]:
trechos, be, idx = montar()
por_id = {t.id: t for t in trechos}
exemplos = ["Qual a largura mínima da faixa livre da calçada?", "Qual a inclinação máxima da rampa de rebaixamento da guia?", "Qual o valor em reais da multa por estacionar na calçada?"]
for q in exemplos:
    r = pipelines.responder_rag(q, idx, be)
    print("PERGUNTA:", q)
    print("JSON bruto:", r.bruto)
    print("  json_valido:", r.json_valido, "| recusou:", r.recusou)
    for i in r.trecho_ids:
        t = por_id.get(i)
        print("  fonte:", t.citacao() if t else f"{i} (INEXISTENTE)")
    print()

PERGUNTA: Qual a largura mínima da faixa livre da calçada?
JSON bruto: { "resposta": "A largura mínima da faixa livre da calçada é de 1,20 m, podendo ser reduzida a 0,90 m apenas em casos excepcionais.", "trecho_ids": [ "T13", "T16", "T17" ], "recusou": false }
  json_valido: True | recusou: False
  fonte: Instituto de Engenharia, palestra ABNT NBR 16537 / NBR 9050:2020 calçada, NBR 9050:2020, calçada
  fonte: AGU, Laudo de acessibilidade PR/Curitiba (cita itens da NBR 9050/2020), NBR 9050/2020, item 6.12.7.3.2
  fonte: Sudeco, Declaração de Conformidade em Acessibilidade (lista de verificação NBR 9050), NBR 9050, item 6.12.7.3

PERGUNTA: Qual a inclinação máxima da rampa de rebaixamento da guia?
JSON bruto: {
  "resposta": "A inclinação máxima da rampa de rebaixamento da guia é de 8,33%.",
  "trecho_ids": ["T18"],
  "recusou": false
}
  json_valido: True | recusou: False
  fonte: CREA-RN, Manual do Síndico, parte 2 (calçadas e travessias), Travessias e guias rebaixadas

PERGUNTA: Qual

In [4]:
# Taxas em todos os casos: JSON válido, citação válida, recusa
casos = kb.carregar_casos()
linhas = []
for c in casos:
    r = pipelines.responder_rag(c["pergunta"], idx, be)
    linhas.append({"id": c["id"], "tipo": c["tipo"], "json_valido": r.json_valido, "recusou": r.recusou,
                   "citacao_valida": all(i in por_id for i in r.trecho_ids), "resposta": r.resposta})
d = pd.DataFrame(linhas)
print(f"JSON válido: {d.json_valido.mean():.0%} | citações válidas: {d.citacao_valida.mean():.0%}")
print(f"recusou em 'não consta': {d[d.tipo=='nao_consta'].recusou.mean():.0%} | recusou indevidamente em 'consta': {d[d.tipo=='consta'].recusou.mean():.0%}")
d[["id", "tipo", "recusou", "resposta"]]

JSON válido: 100% | citações válidas: 100%
recusou em 'não consta': 100% | recusou indevidamente em 'consta': 4%


,id,tipo,recusou,resposta
0,C01,consta,False,"A largura mínima da faixa livre da calçada, segundo a NBR 9050, é de 1,20 m, sendo 1,50 m o recomendado."
1,C02,consta,False,"A faixa livre deve manter no mínimo 1,20 m de largura livre."
2,C03,consta,False,A inclinação transversal máxima da faixa livre é de até 3%.
3,C04,consta,False,"A largura mínima recomendada para a faixa de serviço da calçada é de 0,70 m."
4,C05,consta,False,"A calçada se divide em três faixas de uso: faixa de serviço, faixa livre (passeio) e faixa de acesso."
5,C06,consta,False,"A faixa de serviço serve para acomodar o mobiliário, os canteiros, as árvores e os postes de iluminação ou sinalização."
6,C07,consta,False,"A largura recomendada para a faixa livre é de 1,50 m."
7,C08,consta,False,"A faixa de acesso pode existir em calçadas com largura superior a 2,00 m."
8,C09,consta,False,"A largura mínima da rampa de rebaixamento da guia é de 1,20 m."
9,C10,consta,False,"A inclinação máxima da rampa de rebaixamento da guia é de 8,33%."


**Leitura:** o RAG simples sempre recebe 3 trechos, mesmo quando nenhum serve. Em casos `nao_consta` o risco é o modelo "completar" com o trecho mais parecido. É isso que o agente (notebook 05) ataca.